<a href="https://colab.research.google.com/github/hannahandkush/Coursework/blob/main/Assignment2MLCrossValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Assignment 2

Compare two approaches for cross-validation: the standard KFold and TimeSeriesSplit. The script below estimates the accuracy (R2 between observed and predicted) for those two approaches.

# 0. Organise the Data: **response variable, predictors, imputation**

In [64]:
#Prompt: remove any unused packages or libraries

import numpy as np
import pandas as pd

import sklearn
from sklearn.model_selection import (
    train_test_split,
    KFold,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score

In [4]:
# Load the dataset
df = pd.read_csv('/content/Aquifer_Petrignano.csv')

df.shape

(5223, 8)

In [6]:
df.describe()


,Rainfall_Bastia_Umbra,Depth_to_Groundwater_P24,Depth_to_Groundwater_P25,Temperature_Bastia_Umbra,Temperature_Petrignano,Volume_C10_Petrignano,Hydrometry_Fiume_Chiascio_Petrignano
count,4199.000000,5168.000000,5184.000000,4199.000000,4199.000000,5025.000000,4199.000000
mean,1.556633,-26.263723,-25.692926,15.030293,13.739081,-29043.296726,2.372517
std,5.217923,3.319858,3.214165,7.794871,7.701369,4751.864371,0.589088
min,0.000000,-34.470000,-33.710000,-3.700000,-4.200000,-45544.896000,0.000000
25%,0.000000,-28.250000,-27.620000,8.800000,7.700000,-31678.560000,2.100000
50%,0.000000,-25.990000,-25.540000,14.700000,13.500000,-28689.120000,2.400000
75%,0.100000,-23.820000,-23.430000,21.400000,20.000000,-26218.080000,2.700000
max,67.300000,-19.660000,-19.100000,33.000000,31.100000,0.000000,4.100000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5223 entries, 0 to 5222
Data columns (total 8 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   Date                                  5223 non-null   object 
 1   Rainfall_Bastia_Umbra                 4199 non-null   float64
 2   Depth_to_Groundwater_P24              5168 non-null   float64
 3   Depth_to_Groundwater_P25              5184 non-null   float64
 4   Temperature_Bastia_Umbra              4199 non-null   float64
 5   Temperature_Petrignano                4199 non-null   float64
 6   Volume_C10_Petrignano                 5025 non-null   float64
 7   Hydrometry_Fiume_Chiascio_Petrignano  4199 non-null   float64
dtypes: float64(7), object(1)
memory usage: 326.6+ KB


In [8]:
df.isnull().sum().sum()

np.int64(4388)

### Missing gaps identified, exploration required to determin fill method (imputation)

I found there were sections of consecutive rows of null entries in termperature and volume

In [10]:
# PROMPT: Explain error: The original error occurs because the 'by' argument of groupby must be 1-dimensional.
# (~df.isnull()).cumsum() produces a DataFrame, which is 2-dimensional.

# If the intention is to find consecutive runs of nulls for each column independently,
# you can apply the logic to each column:
result = df.apply(lambda col: col.isnull().astype(int).groupby((~col.isnull()).cumsum()).sum())
print(result)


      Date  Rainfall_Bastia_Umbra  Depth_to_Groundwater_P24  \
0      NaN                 1024.0                       NaN   
1      0.0                    0.0                       0.0   
2      0.0                    0.0                       0.0   
3      0.0                    0.0                       0.0   
4      0.0                    0.0                       0.0   
...    ...                    ...                       ...   
5219   0.0                    NaN                       NaN   
5220   0.0                    NaN                       NaN   
5221   0.0                    NaN                       NaN   
5222   0.0                    NaN                       NaN   
5223   0.0                    NaN                       NaN   

      Depth_to_Groundwater_P25  Temperature_Bastia_Umbra  \
0                          NaN                    1024.0   
1                          0.0                       0.0   
2                          0.0                       0.0   
3  

In [11]:
(df.isnull() & df.notnull().shift()).sum()

,0
Date,0
Rainfall_Bastia_Umbra,0
Depth_to_Groundwater_P24,10
Depth_to_Groundwater_P25,7
Temperature_Bastia_Umbra,0
Temperature_Petrignano,0
Volume_C10_Petrignano,1
Hydrometry_Fiume_Chiascio_Petrignano,0


### Missing gaps identified with time stamps

In [13]:
def get_missing_gaps(df):
    all_gaps = []

    for col in df.columns:
        is_nan = df[col].isnull()

        # detect starts and ends of gaps (robust to edge cases)
        starts = df.index[is_nan & ~is_nan.shift(1, fill_value=False)]
        ends   = df.index[is_nan & ~is_nan.shift(-1, fill_value=False)]

        for s, e in zip(starts, ends):
            all_gaps.append({
                "column": col,
                "start": s,
                "end": e,
                "length_days": (e - s).days + 1
            })

    return pd.DataFrame(all_gaps)

In [14]:
df = df.set_index("Date")
df.index = pd.to_datetime(df.index)
df = df.sort_index()

/tmp/ipykernel_12378/741838274.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df.index = pd.to_datetime(df.index)


In [18]:
gaps_df = get_missing_gaps(df)
gaps_df.sort_values(["column", "start"])

,column,start,end,length_days
1,Depth_to_Groundwater_P24,2006-06-09,2006-06-09,1
2,Depth_to_Groundwater_P24,2006-10-15,2006-10-15,1
3,Depth_to_Groundwater_P24,2006-12-14,2006-12-14,1
4,Depth_to_Groundwater_P24,2007-09-02,2007-09-02,1
5,Depth_to_Groundwater_P24,2007-12-20,2007-12-20,1
6,Depth_to_Groundwater_P24,2008-05-17,2008-05-27,11
7,Depth_to_Groundwater_P24,2012-06-26,2012-07-02,7
8,Depth_to_Groundwater_P24,2013-03-13,2013-04-02,21
9,Depth_to_Groundwater_P24,2019-03-18,2019-03-24,7
10,Depth_to_Groundwater_P24,2020-03-29,2020-04-01,4


### **Imputation**

In [40]:
# Prompt: is it statistically sound to treat the gaps differently depending on size? what should I do for the big gaps?

# 1. Drop big missing period
df = df.loc["2009-01-01":]

# 2. Rainfall: fill with 0, not needed because rainfaill values present from 2009 onwards
#df["Rainfall_Bastia_Umbra"] = df["Rainfall_Bastia_Umbra"].fillna(0)

# 3. Interpolate smooth variables, removed "Temperature_Bastia_Umbra", "Temperature_Petrignano" and "Hydrometry_Fiume_Chiascio_Petrignano" because no gaps after 2009
cols_interp = [
    "Depth_to_Groundwater_P24",
    "Depth_to_Groundwater_P25",
    "Volume_C10_Petrignano"
]

df[cols_interp] = df[cols_interp].interpolate(method="time")

# 4. Optional: clean leftovers
df = df.fillna(method="ffill")

/tmp/ipykernel_12378/3572989880.py:19: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method="ffill")


In [41]:
print(df.index.min())
print(df.index.max())
print(len(df))

2009-01-01 00:00:00
2020-06-30 00:00:00
4199


In [42]:
df.isnull().sum().sum()

#no more missing values

np.int64(0)

gaps dealt with,
np.int64(0)

## Further pre-processing steps to suit task requirements



*   daily --> monthly
*   features generated with lag of 1 and 2 months
*   seasonality generated (to link month 12 to month 1, looping year
*   feature matrix created
*   test train split, separating final 20% data in the time series (not shuffled)



In [43]:
import numpy as np

# Prompt

# 1. Resample daily → monthly means
df_monthly = df.resample("MS").mean()

# 2. Create lagged features (t-1 and t-2 months)
lag1 = df_monthly.shift(1).add_suffix("_lag1")
lag2 = df_monthly.shift(2).add_suffix("_lag2")

# 3. Optional: seasonality
df_monthly["month_sin"] = np.sin(2 * np.pi * df_monthly.index.month / 12)
df_monthly["month_cos"] = np.cos(2 * np.pi * df_monthly.index.month / 12)

# 4. Combine into feature matrix
df_features = pd.concat([lag1, lag2], axis=1)

# Add seasonality columns (these are current-month, not lagged — that's fine)
df_features["month_sin"] = df_monthly["month_sin"]
df_features["month_cos"] = df_monthly["month_cos"]

# 5. Target: DP25 at current month
df_features["target"] = df_monthly["Depth_to_Groundwater_P25"]

# 6. Drop NaNs from lagging and sort chronologically
df_features = df_features.dropna().sort_index()

# 7. Split X and y
X = df_features.drop(columns=["target"])
y = df_features["target"]

print(f"X shape: {X.shape}")
print(f"Date range: {X.index.min()} → {X.index.max()}")
print(f"Features: {list(X.columns)}")

X shape: (136, 16)
Date range: 2009-03-01 00:00:00 → 2020-06-01 00:00:00
Features: ['Rainfall_Bastia_Umbra_lag1', 'Depth_to_Groundwater_P24_lag1', 'Depth_to_Groundwater_P25_lag1', 'Temperature_Bastia_Umbra_lag1', 'Temperature_Petrignano_lag1', 'Volume_C10_Petrignano_lag1', 'Hydrometry_Fiume_Chiascio_Petrignano_lag1', 'Rainfall_Bastia_Umbra_lag2', 'Depth_to_Groundwater_P24_lag2', 'Depth_to_Groundwater_P25_lag2', 'Temperature_Bastia_Umbra_lag2', 'Temperature_Petrignano_lag2', 'Volume_C10_Petrignano_lag2', 'Hydrometry_Fiume_Chiascio_Petrignano_lag2', 'month_sin', 'month_cos']


### **Predictors**

In [44]:
# predictors are monthly averages for all variables for the 2 months preceding
df_features.head()

,Rainfall_Bastia_Umbra_lag1,Depth_to_Groundwater_P24_lag1,Depth_to_Groundwater_P25_lag1,Temperature_Bastia_Umbra_lag1,Temperature_Petrignano_lag1,Volume_C10_Petrignano_lag1,Hydrometry_Fiume_Chiascio_Petrignano_lag1,Rainfall_Bastia_Umbra_lag2,Depth_to_Groundwater_P24_lag2,Depth_to_Groundwater_P25_lag2,Temperature_Bastia_Umbra_lag2,Temperature_Petrignano_lag2,Volume_C10_Petrignano_lag2,Hydrometry_Fiume_Chiascio_Petrignano_lag2,month_sin,month_cos,target
Date,,,,,,,,,,,,,,,,,
2009-03-01,0.500000,-29.918214,-29.350000,5.125000,4.792857,-25641.174857,2.546429,0.203226,-31.278387,-30.503871,4.964516,4.945161,-26695.119484,2.387097,1.000000e+00,6.123234e-17,-28.502903
2009-04-01,1.103226,-29.041613,-28.502903,9.254839,8.716129,-28374.874839,2.490323,0.500000,-29.918214,-29.350000,5.125000,4.792857,-25641.174857,2.546429,8.660254e-01,-5.000000e-01,-27.736333
2009-05-01,0.543333,-28.229000,-27.736333,13.806667,13.250000,-28167.667200,2.403333,1.103226,-29.041613,-28.502903,9.254839,8.716129,-28374.874839,2.490323,5.000000e-01,-8.660254e-01,-27.478065
2009-06-01,0.922581,-28.045806,-27.478065,19.809677,18.812903,-32187.288774,2.109677,0.543333,-28.229000,-27.736333,13.806667,13.250000,-28167.667200,2.403333,1.224647e-16,-1.000000e+00,-27.522000
2009-07-01,0.010000,-28.167333,-27.522000,21.203333,20.273333,-31514.832000,2.053333,0.922581,-28.045806,-27.478065,19.809677,18.812903,-32187.288774,2.109677,-5.000000e-01,-8.660254e-01,-27.721613


## 1. The Hard Split

### **Independent test data set**

In [28]:
#shuffle=false to ensure test data is the final 20% of the time series,
X_train_full, X_test, y_train_full, y_test = train_test_split(    X, y, test_size=0.2, shuffle=False)

In [29]:
print(f"Train: {X_train_full.index.min()} → {X_train_full.index.max()}")
print(f"Test:  {X_test.index.min()} → {X_test.index.max()}")

Train: 2009-03-01 00:00:00 → 2018-02-01 00:00:00
Test:  2018-03-01 00:00:00 → 2020-06-01 00:00:00


Training set on the left of the diagram contains data from dates:

2009-03-01 → 2018-02-01

Test step on the right of the diagram contains data from dates:

2018-03-01 → 2020-06-01

![](https://github.com/isa-ulisboa/greends-pml/blob/main/docs/holdout_method_fig62.png?raw=true)

# 2. Define the two **Cross Validation** Strategies: **folds, data leakage**

#### **Kfold** for cv_naive uses shuffled sections leading to **data leakage** whereas cv_temporal uses  a **timeseriessplit** which keeps an **independent test data set**

In [30]:
from sklearn.model_selection import KFold, TimeSeriesSplit
cv_naive = KFold(n_splits=5, shuffle=True, random_state=42)
cv_temporal = TimeSeriesSplit(n_splits=5)

Kfold naive split method

![](https://github.com/isa-ulisboa/greends-pml/blob/main/docs/kfold_validation_fig_63.png?raw=true)

Time Series Split method

Fold 1: [■■□□□□□□]   train on first chunk, validate on next

Fold 2: [■■■□□□□□]   train grows, validate moves forward

Fold 3: [■■■■□□□□]

Fold 4: [■■■■■□□□]

Fold 5: [■■■■■■□□]

# 3. Fixed Model **Cross-Validation**: **hyper paramters**

In [57]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])

# hyperparameters fixed, max_depth=10

In [58]:
from sklearn.model_selection import cross_val_score

# Strategy 1: Naive (Shuffled)
# This will likely report a very high R2 because of leakage.
scores_naive = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_naive, scoring='r2')

In [59]:
# Strategy 2: Temporal (Sequential)
# This will report a lower, more honest R2.
scores_temporal = cross_val_score(pipe, X_train_full, y_train_full, cv=cv_temporal, scoring='r2')

In [60]:
print(f"Naive CV R2:    {scores_naive.mean():.4f} (+/- {scores_naive.std():.4f})")
print(f"Temporal CV R2: {scores_temporal.mean():.4f} (+/- {scores_temporal.std():.4f})")

Naive CV R2:    0.9275 (+/- 0.0205)
Temporal CV R2: -0.9542 (+/- 2.5934)


In [61]:
# Train on all training data and test on the unseen "future"
pipe.fit(X_train_full, y_train_full)
final_test_r2 = r2_score(y_test, pipe.predict(X_test))

print(f"\nActual Test R2 on 2021 Data: {final_test_r2:.4f}")


Actual Test R2 on 2021 Data: -0.1402


### **Data leakage, model selection**

A good R is close to 1


Naive CV = 0.93
overfitting because of data leakage, it is training using future values.

Temporal CV = -0.95
memorised past patterns that don't transfer forward, memorising noise

Actual Test = -0.14
memorised past patterns that don't transfer forward, memorising noise

Naive CV is much higher than the other two because it doesn't exclude future values from the training phase

# 4. **Model Selection** & Evaluation: **hyper parameters**

In [48]:
def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):

    # STEP A: Define the Pipeline (Hint: Use 'scaler' and 'regressor' as step names)
    # Prompt: how do i know which type of regression to use here i.e. AdalineSGD, SGDRegressor
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])

    # STEP B: Define the Hyperparameter Grid
    param_grid = {
        "regressor__max_depth": [1, 2, 3, 4, 5,7,10,15,30]
    }

    # STEP C: Initialize and Fit GridSearchCV
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring="r2"
    )
    grid.fit(X_train, y_train)

    # Removed redundant grid.fit(X_train, y_train) call

    # STEP D: Final Evaluation
    # Use the 'best_estimator_' to predict the unseen X_test
    y_pred = grid.predict(X_test)
    est_r2 = r2_score(y_test, y_pred)

    print(f"\n===== Results for: {name} =====")
    print(f"Best Parameters found: {grid.best_params_}")
    print(f"Internal CV Score (R2): {grid.best_score_:.4f}")
    print(f"Independent Test Score (R2): {est_r2:.4f}") # Changed test_r2 to est_r2

    return grid

In [49]:
#Prompt: print grid

grid_naive = evaluate_model_selection(
    X_train=X_train_full,
    y_train=y_train_full,
    X_test=X_test,
    y_test=y_test,
    cv_strategy=cv_naive,
    name="Naive CV"
)


===== Results for: Naive CV =====
Best Parameters found: {'regressor__max_depth': 7}
Internal CV Score (R2): 0.9278
Independent Test Score (R2): -0.5024


In [50]:
grid_temporal = evaluate_model_selection(
    X_train=X_train_full,
    y_train=y_train_full,
    X_test=X_test,
    y_test=y_test,
    cv_strategy=cv_temporal,
    name="Temporal CV"
)


===== Results for: Temporal CV =====
Best Parameters found: {'regressor__max_depth': 3}
Internal CV Score (R2): 0.1102
Independent Test Score (R2): 0.2320



===== Results for: Naive CV =====
Best Parameters found: {'regressor__max_depth': 7}

Internal CV Score (R2): 0.9278

Independent Test Score (R2): -0.5024


===== Results for: Temporal CV =====
Best Parameters found: {'regressor__max_depth': 3}

Internal CV Score (R2): 0.1102

Independent Test Score (R2): 0.2320


# Conclusion: **Model Selection**

For Naive CV the model still overfits due to data leakage because **Standard KFold** assumes data is independent and possible to shuffle. At the optimal max depth of 5, the internal CV score of 0.9278 appears good but is meaningless because folds include data from future months, so this score cannot be trusted. The true performance is the independent test score of -0.5024, which is substantially lower and reflects what the model actually achieves on unseen future data.

For Temporal CV the best hyperparameter for maximum tree depth is 3. This prevents it learning noise and therefore overfitting. The internal CV score of 0.1102 is low partly because the earliest **TimeSeriesSplit folds** train on very little data, which disadvantages the model. The independent test score of 0.2320 is actually higher than the internal CV score, which is unusual (it performs better on the unseen test sample) this suggests the most recent months in the test set may be slightly more predictable than the earlier periods used in the CV folds.

The gap between the naive internal CV score (0.9278 ) and the temporal internal CV score (0.1102) idicates the degree of impact that **data leakage** has.

At best the model captures under 12% of the variance in future groundwater depth, which points to two months of lagged features and a decision tree being too simple an approach to model this system well.